In [2]:
import string
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

dataset = load_dataset("imdb")

train_df = pd.DataFrame({
    'review': dataset['train']['text'],
    'label':  dataset['train']['label']
})

test_df = pd.DataFrame({
    'review': dataset['test']['text'],
    'label':  dataset['test']['label']
})

print(f"Train size: {len(train_df)}")
print(f"Test size:  {len(test_df)}")
train_df.head()

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train size: 25000
Test size:  25000


,review,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [3]:
def preprocess_text(text):
    text = text.lower()
    text = text.replace("<br />", " ")
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    return tokens

train_df['tokens'] = train_df['review'].apply(preprocess_text)
test_df['tokens']  = test_df['review'].apply(preprocess_text)

train_df.head()

,review,label,tokens
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,"[i, rented, i, am, curiousyellow, from, my, vi..."
1,"""I Am Curious: Yellow"" is a risible and preten...",0,"[i, am, curious, yellow, is, a, risible, and, ..."
2,If only to avoid making this type of film in t...,0,"[if, only, to, avoid, making, this, type, of, ..."
3,This film was probably inspired by Godard's Ma...,0,"[this, film, was, probably, inspired, by, goda..."
4,"Oh, brother...after hearing about this ridicul...",0,"[oh, brotherafter, hearing, about, this, ridic..."


In [4]:
def build_vocabulary(tokens_series):
    vocab = {}
    index = 0
    for tokens in tokens_series:
        for word in tokens:
            if word not in vocab:
                vocab[word] = index
                index += 1
    return vocab

vocab = build_vocabulary(train_df['tokens'])
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 112011


In [7]:
def build_bow_matrix(tokens_series, vocab):
    tokens_list = list(tokens_series)
    matrix = np.zeros((len(tokens_list), len(vocab)), dtype=np.int16)
    for i, tokens in enumerate(tokens_list):
        for word in tokens:
            if word in vocab:
                matrix[i][vocab[word]] += 1
    return matrix

train_bow = build_bow_matrix(train_df['tokens'], vocab)
test_bow  = build_bow_matrix(test_df['tokens'],  vocab)

In [6]:
def compute_prior(df):
    total     = len(df)
    prior_pos = len(df[df['label'] == 1]) / total
    prior_neg = len(df[df['label'] == 0]) / total
    return prior_pos, prior_neg

prior_pos, prior_neg = compute_prior(train_df)
print(f"P(positive) = {prior_pos}")
print(f"P(negative) = {prior_neg}")

P(positive) = 0.5
P(negative) = 0.5
